In [12]:
!pip install langchain langchain-community langchain-google-genai chromadb sentence-transformers -q


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import getpass

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Enter Gemini API Key:  ········


In [3]:
documents = [
    {"page_content": "Machine Learning is a subset of AI that uses algorithms to learn from data.",
     "metadata": {"category": "AI", "source": "ml_doc1"}},
    {"page_content": "HR policies include employee benefits, leave management, and performance reviews.",
     "metadata": {"category": "HR", "source": "hr_doc1"}},
    {"page_content": "Deep Learning uses neural networks with multiple layers to process complex data.",
     "metadata": {"category": "AI", "source": "ml_doc2"}},
    {"page_content": "Employee onboarding process includes documentation, training, and orientation.",
     "metadata": {"category": "HR", "source": "hr_doc2"}},
    {"page_content": "Supervised learning requires labeled training data for model training.",
     "metadata": {"category": "AI", "source": "ml_doc3"}}
]

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_texts(
    texts=[doc["page_content"] for doc in documents],
    embedding=embeddings,
    metadatas=[doc["metadata"] for doc in documents]
)

C:\Users\User\AppData\Local\Temp\ipykernel_14904\31082186.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3295.97it/s]


In [5]:
print("="*50)
print("1. HYBRID SEARCH (Similarity Search)")
print("="*50)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

query = "What is Machine Learning?"
results = retriever.invoke(query)

print(f"\nQuery: {query}")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content[:100]}... (Category: {doc.metadata.get('category', 'N/A')})")

1. HYBRID SEARCH (Similarity Search)

Query: What is Machine Learning?
1. Machine Learning is a subset of AI that uses algorithms to learn from data.... (Category: AI)
2. Deep Learning uses neural networks with multiple layers to process complex data.... (Category: AI)
3. Supervised learning requires labeled training data for model training.... (Category: AI)


In [6]:
print("\n" + "="*50)
print("2. METADATA FILTERING (HR Category)")
print("="*50)

filtered_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "filter": {"category": "HR"}}
)

filtered_results = filtered_retriever.invoke("employee policies")
for i, doc in enumerate(filtered_results, 1):
    print(f"{i}. {doc.page_content} (Category: {doc.metadata.get('category')})")


2. METADATA FILTERING (HR Category)
1. HR policies include employee benefits, leave management, and performance reviews. (Category: HR)
2. Employee onboarding process includes documentation, training, and orientation. (Category: HR)


In [7]:
print("\n" + "="*50)
print("3. RAG EVALUATION")
print("="*50)

llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=filtered_retriever
)

question = "What are the HR policies?"
answer = rag_chain.invoke(question)
print(f"\nQuestion: {question}")
print(f"Answer: {answer}")


3. RAG EVALUATION

Question: What are the HR policies?
Answer: {'query': 'What are the HR policies?', 'result': 'Based on the provided context, HR policies include:\n* Employee benefits\n* Leave management\n* Performance reviews'}


In [8]:
print("\n" + "="*50)
print("4. RAG EVALUATION METRICS")
print("="*50)

context_docs = filtered_retriever.invoke(question)
print(f"- Number of retrieved documents: {len(context_docs)}")
print(f"- Retrieved document categories: {[doc.metadata.get('category') for doc in context_docs]}")
print(f"- Answer: {answer['result'][:150]}...")


4. RAG EVALUATION METRICS
- Number of retrieved documents: 2
- Retrieved document categories: ['HR', 'HR']
- Answer: Based on the provided context, HR policies include:
* Employee benefits
* Leave management
* Performance reviews...


In [9]:
class HybridSearch:
    def __init__(self, documents):
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        self.vectorstore = Chroma.from_texts(
            texts=documents,
            embedding=self.embeddings
        )

    def similarity_search(self, query, k=3):
        retriever = self.vectorstore.as_retriever(
            search_type="similarity", search_kwargs={"k": k}
        )
        return retriever.invoke(query)

    def mmr_search(self, query, k=3):
        retriever = self.vectorstore.as_retriever(
            search_type="mmr", search_kwargs={"k": k, "fetch_k": 10}
        )
        return retriever.invoke(query)

    def threshold_search(self, query, k=3, threshold=0.7):
        retriever = self.vectorstore.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs={"k": k, "score_threshold": threshold}
        )
        return retriever.invoke(query)

    def display_results(self, query, search_type="similarity", k=3):
        print(f"\n{'='*50}")
        print(f"Search Type: {search_type.upper()}")
        print(f"Query: '{query}'")
        print('='*50)

        if search_type == "similarity":
            results = self.similarity_search(query, k)
        elif search_type == "mmr":
            results = self.mmr_search(query, k)
        else:
            results = self.threshold_search(query, k)

        if not results:
            print("No results found!")
            return

        for i, doc in enumerate(results, 1):
            print(f"\nResult #{i}")
            print(f"   Content: {doc.page_content}")
        return results

In [10]:
documents_2 = [
    "Pakistan is a country in South Asia with a rich history.",
    "Islamabad is the capital city of Pakistan.",
    "Lahore is famous for its culture, food, and historical places.",
    "Karachi is the largest city and economic hub of Pakistan.",
    "Pakistan's population is over 240 million people.",
    "The independence of Pakistan was achieved on 14 August 1947.",
    "Urdu and English are the official languages of Pakistan."
]

search = HybridSearch(documents_2)

queries = [
    "Which city is the capital?",
    "Tell me about major cities in Pakistan",
    "When did Pakistan become independent?"
]

for q in queries:
    search.display_results(q, "similarity", k=3)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1318.15it/s]



Search Type: SIMILARITY
Query: 'Which city is the capital?'

Result #1
   Content: Islamabad is the capital city of Pakistan.

Result #2
   Content: Karachi is the largest city and economic hub of Pakistan.

Result #3
   Content: Lahore is famous for its culture, food, and historical places.

Search Type: SIMILARITY
Query: 'Tell me about major cities in Pakistan'

Result #1
   Content: Karachi is the largest city and economic hub of Pakistan.

Result #2
   Content: Islamabad is the capital city of Pakistan.

Result #3
   Content: Lahore is famous for its culture, food, and historical places.

Search Type: SIMILARITY
Query: 'When did Pakistan become independent?'

Result #1
   Content: The independence of Pakistan was achieved on 14 August 1947.

Result #2
   Content: Pakistan is a country in South Asia with a rich history.

Result #3
   Content: Islamabad is the capital city of Pakistan.


In [11]:
sample_pdf_text = """
Name: John

Email:
john@gmail.com

Skills:
Python
SQL
Django
"""

extracted_json = {
    "name": "John",
    "email": "john@gmail.com",
    "skills": ["Python", "SQL", "Django"]
}
print(extracted_json)

{'name': 'John', 'email': 'john@gmail.com', 'skills': ['Python', 'SQL', 'Django']}
